# 05 · Datasets: preparar, limpiar, anotar y estructurar (≈1.5 h)

La oferta lo pide **dos veces** ("preparando, limpiando, estructurando y anotando datasets"). En la práctica
es donde más se gana: **un dataset mejor suele dar más que un modelo mejor**. Objetivos:
- Conocer los formatos (COCO, YOLO, MOT) y convertir entre ellos ✍️.
- Saber organizar un proceso de anotación (guía, herramientas, pre-anotación, QA).
- Detectar y limpiar errores de etiquetas ✍️.
- Evitar **data leakage** al dividir datos de vídeo (lo preguntan mucho).
- Construir un dataset YOLO a partir de un vídeo y entrenar con él.

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
from pathlib import Path
import json, shutil, random
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## 1. Ciclo de vida de un dataset de visión

1. **Definir la tarea y la guía de anotación** (lo más infravalorado). Ejemplos de decisiones que hay que dejar
   por escrito: ¿la caja de una persona ocluida cubre solo lo visible o la persona entera (*amodal*)? ¿El árbitro
   es "persona" o clase aparte? ¿Se anota el balón si está borroso? ¿Gente del público? Sin guía, cada
   anotador hace una cosa → ruido en las etiquetas → techo de precisión.
2. **Muestreo de frames**: en vídeo, frames consecutivos son casi iguales. Anotar todos es caro e inútil;
   se muestrea (1 de cada N, o por diversidad / cambios de escena) y se priorizan situaciones difíciles.
3. **Pre-anotación** (*model-assisted labeling*): un modelo (YOLO grande, SAM para máscaras) propone y el humano
   corrige. Muchísimo más rápido.
4. **Anotación** con herramienta:
   - **CVAT** (open source, auto-hospedable): cajas, polígonos, keypoints, y sobre todo **tracks en vídeo con
     interpolación** entre keyframes (anotas cada 10 frames y CVAT interpola) → ideal para GT de tracking (IDs).
     Integra SAM y modelos para auto-anotación. Exporta COCO, YOLO, MOT, etc.
   - **Roboflow**: SaaS; anotación, versionado, augmentations, exportación a cualquier formato, entrenamiento.
   - Label Studio, FiftyOne (exploración y curación, no tanto anotación).
5. **QA**: revisión por muestreo, doble anotación de un subconjunto y medir acuerdo (IoU entre anotadores),
   scripts de validación automática (sección 4), revisar las imágenes donde el modelo y la etiqueta discrepan.
6. **Split** sin leakage (sección 5) y **versionado** (DVC, Roboflow versions, o al menos un hash + manifest):
   cada experimento debe saber con qué versión exacta de datos se entrenó.
7. **Iterar** (*data-centric AI*): entrenar → análisis de errores → anotar más de lo que falla → repetir.
   **Active learning**: priorizar para anotar los frames donde el modelo duda (confianzas intermedias), donde
   hay muchas detecciones solapadas o donde el tracker hace ID switches.

## 2. Formatos que hay que conocer

**YOLO (Ultralytics)** — un `.txt` por imagen, misma ruta cambiando `images/` → `labels/`:
```
0 0.512 0.430 0.061 0.210        # clase cx cy w h, normalizados a [0,1]
```
Segmentación: `clase x1 y1 x2 y2 ...` (polígono normalizado). Pose: `clase cx cy w h kx1 ky1 v1 ...`.
Imágenes sin objetos: `.txt` vacío (o ausente) → **imágenes de fondo**, útiles para bajar FP (≈1–10%).

**COCO** — un único JSON:
```json
{"images": [{"id": 1, "file_name": "f0001.jpg", "width": 1920, "height": 1080}],
 "annotations": [{"id": 7, "image_id": 1, "category_id": 1, "bbox": [x, y, w, h],
                  "area": 1234.0, "iscrowd": 0, "segmentation": [[x1, y1, ...]]}],
 "categories": [{"id": 1, "name": "person"}]}
```
¡`bbox` en COCO es **xywh en píxeles** (esquina sup-izq)! y los `category_id` no tienen por qué empezar en 0.

**MOTChallenge** (tracking) — CSV `gt.txt` / resultados:
```
frame, id, x, y, w, h, conf, clase, visibilidad      # frame empieza en 1, caja xywh en píxeles
1, 3, 794.2, 47.5, 71.2, 174.8, 1, 1, 0.8
```
Es el formato que esperan TrackEval y `motmetrics` (notebook 06).

**Pascal VOC**: un XML por imagen con cajas xyxy (antiguo, aún aparece).

## 3. ✍️ EJERCICIO: COCO → YOLO

Escribe `coco_to_yolo(coco: dict, out_dir: Path)` que cree un `.txt` por imagen. Cuidado con:
mapear `category_id` → índices 0..C-1, convertir xywh-px → cxcywh-normalizado, ignorar `iscrowd=1`,
y crear también los `.txt` vacíos de imágenes sin anotaciones.

In [ ]:
def coco_to_yolo(coco, out_dir):
    # TU CÓDIGO AQUÍ → devuelve el dict de nombres {idx: nombre}
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def coco_to_yolo(coco, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cats = sorted(coco["categories"], key=lambda c: c["id"])
    cat2idx = {c["id"]: i for i, c in enumerate(cats)}
    imgs = {im["id"]: im for im in coco["images"]}
    lines = {im_id: [] for im_id in imgs}
    for a in coco["annotations"]:
        if a.get("iscrowd", 0):
            continue
        im = imgs[a["image_id"]]
        x, y, w, h = a["bbox"]
        W, H = im["width"], im["height"]
        cx, cy = (x + w / 2) / W, (y + h / 2) / H
        lines[a["image_id"]].append(f"{cat2idx[a['category_id']]} {cx:.6f} {cy:.6f} {w / W:.6f} {h / H:.6f}")
    for im_id, ls in lines.items():
        (out_dir / (Path(imgs[im_id]["file_name"]).stem + ".txt")).write_text("\n".join(ls))
    return {i: c["name"] for i, c in enumerate(cats)}

In [ ]:
coco_demo = {
    "images": [{"id": 10, "file_name": "a.jpg", "width": 1000, "height": 500},
               {"id": 11, "file_name": "b.jpg", "width": 1000, "height": 500}],
    "annotations": [{"id": 1, "image_id": 10, "category_id": 1, "bbox": [100, 100, 200, 100], "iscrowd": 0},
                    {"id": 2, "image_id": 10, "category_id": 37, "bbox": [500, 250, 10, 10], "iscrowd": 0},
                    {"id": 3, "image_id": 10, "category_id": 1, "bbox": [0, 0, 1000, 500], "iscrowd": 1}],
    "categories": [{"id": 1, "name": "person"}, {"id": 37, "name": "sports ball"}],
}
tmp = DATA / "tmp_labels"
print(coco_to_yolo(coco_demo, tmp))
print("a.txt:\n" + (tmp / "a.txt").read_text(), "\nb.txt vacío:", repr((tmp / "b.txt").read_text()))
assert (tmp / "a.txt").read_text().splitlines()[0] == "0 0.200000 0.300000 0.200000 0.200000"
shutil.rmtree(tmp)

## 4. ✍️ EJERCICIO: validar y limpiar etiquetas

Errores típicos que hay que cazar automáticamente:
- Cajas fuera de la imagen o con coordenadas fuera de [0,1] → recortar (*clip*) o descartar.
- Cajas degeneradas (w o h ≈ 0) → descartar.
- **Duplicados** (dos cajas casi idénticas sobre el mismo objeto, típico al fusionar anotadores o pre-anotación)
  → IoU > 0.9 con la misma clase.
- Clases fuera de rango.
- Outliers de tamaño/aspecto (una "persona" de 3 px o más ancha que alta ×5) → revisar a mano.

Implementa `clean_labels(rows, n_classes)` con `rows: (N,5)` en formato YOLO; devuelve filas limpias y un
dict con el recuento de cada problema.

In [ ]:
def clean_labels(rows, n_classes, min_wh=1e-3, dup_iou=0.9):
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def _iou_xyxy(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(rb - lt, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-12)

def clean_labels(rows, n_classes, min_wh=1e-3, dup_iou=0.9):
    rows = np.asarray(rows, np.float64).reshape(-1, 5)
    report = {"clase_invalida": 0, "recortadas": 0, "degeneradas": 0, "duplicadas": 0}
    ok = (rows[:, 0] >= 0) & (rows[:, 0] < n_classes) & (rows[:, 0] == rows[:, 0].round())
    report["clase_invalida"] = int((~ok).sum()); rows = rows[ok]
    # a xyxy normalizado, recortar a [0,1]
    xyxy = np.c_[rows[:, 1] - rows[:, 3] / 2, rows[:, 2] - rows[:, 4] / 2, rows[:, 1] + rows[:, 3] / 2, rows[:, 2] + rows[:, 4] / 2]
    clipped = np.clip(xyxy, 0, 1)
    report["recortadas"] = int((np.abs(clipped - xyxy) > 1e-9).any(1).sum())
    wh = clipped[:, 2:] - clipped[:, :2]
    ok = (wh > min_wh).all(1)
    report["degeneradas"] = int((~ok).sum()); clipped, rows = clipped[ok], rows[ok]
    keep = np.ones(len(rows), bool)
    if len(rows) > 1:
        iou = _iou_xyxy(clipped, clipped); np.fill_diagonal(iou, 0)
        same = rows[:, 0][:, None] == rows[:, 0][None, :]
        for i in range(len(rows)):
            if keep[i]:
                d = (iou[i] > dup_iou) & same[i] & keep & (np.arange(len(rows)) > i)
                keep[d] = False
    report["duplicadas"] = int((~keep).sum()); clipped, rows = clipped[keep], rows[keep]
    out = np.c_[rows[:, 0], (clipped[:, 0] + clipped[:, 2]) / 2, (clipped[:, 1] + clipped[:, 3]) / 2,
                clipped[:, 2] - clipped[:, 0], clipped[:, 3] - clipped[:, 1]]
    return out, report

In [ ]:
dirty = [[0, 0.5, 0.5, 0.2, 0.4],
         [0, 0.501, 0.5, 0.2, 0.4],     # duplicada
         [0, 0.98, 0.5, 0.1, 0.2],      # se sale por la derecha
         [1, 0.3, 0.3, 0.0, 0.1],       # degenerada
         [7, 0.3, 0.3, 0.1, 0.1]]       # clase inválida (n_classes=2)
clean, rep = clean_labels(dirty, n_classes=2)
print(rep); print(clean.round(3))

## 5. Data leakage en vídeo (pregunta casi segura)

Si sacas 3.000 frames de 10 partidos y haces `random_split` 80/20 **por frame**, el frame 1501 está en train
y el 1502 en val: son **casi idénticos**. El modelo "memoriza" y la métrica de val sale **inflada**; luego en
un partido nuevo rinde mucho peor.

**Regla: dividir por la unidad que será nueva en producción** → por **vídeo / partido / cámara / estadio / día**.
Si solo tienes un vídeo, por **bloques temporales contiguos con un hueco** (*gap*) entre train y val.
Además, el test debe parecerse a producción (otros estadios, iluminación, nocturno...). Lo mismo aplica a
ReID: **las identidades de test no deben aparecer en train**.

Veámoslo: similitud entre frames según la distancia temporal.

In [ ]:
cap = cv2.VideoCapture(str(DATA / "people.mp4"))
frames = []
while True:
    ok, f = cap.read()
    if not ok: break
    frames.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), (160, 90)).astype(np.float32))
cap.release()
F = np.stack(frames)
gaps = [1, 2, 5, 10, 25, 50, 100, 200]
sims = [np.mean([np.corrcoef(F[i].ravel(), F[i + g].ravel())[0, 1] for i in range(0, len(F) - g, 5)]) for g in gaps]
plt.plot(gaps, sims, "o-"); plt.xscale("log"); plt.xlabel("separación (frames)"); plt.ylabel("correlación de píxeles")
plt.title("Frames cercanos ≈ duplicados → no repartir entre train y val"); plt.grid();

## 6. Construir un dataset YOLO desde un vídeo (pseudo-etiquetado)

Flujo realista de "prueba de concepto rápida":
1. Extraer 1 de cada N frames.
2. **Pre-anotar** con un modelo grande a alta resolución (aquí `yolo11s` a 1280; en real luego un humano revisa
   en CVAT/Roboflow).
3. Limpiar etiquetas, **dividir por bloques temporales con hueco**, escribir `data.yaml`.
4. Entrenar un modelo pequeño (*knowledge distillation* "de pobre": el pequeño aprende del grande en tu dominio).

In [ ]:
ds = DATA / "people_ds"
if ds.exists(): shutil.rmtree(ds)
teacher = YOLO(str(DATA / "yolo11s.pt"))
cap = cv2.VideoCapture(str(DATA / "people.mp4"))
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
STEP, GAP = 3, 30
split_at = int(n_total * 0.75)
stats = {"train": 0, "val": 0}
for i in range(n_total):
    ok, f = cap.read()
    if not ok: break
    if i % STEP: continue
    if split_at - GAP <= i < split_at: continue          # hueco temporal entre train y val
    split = "train" if i < split_at else "val"
    r = teacher(f, imgsz=1280, conf=0.4, classes=[0], device=DEVICE, verbose=False)[0]
    rows = np.c_[np.zeros(len(r.boxes)), r.boxes.xywhn.cpu().numpy()]
    rows, _ = clean_labels(rows, n_classes=1)
    (ds / "images" / split).mkdir(parents=True, exist_ok=True); (ds / "labels" / split).mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(ds / "images" / split / f"f{i:05d}.jpg"), f)
    (ds / "labels" / split / f"f{i:05d}.txt").write_text("\n".join(f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in rows))
    stats[split] += 1
cap.release()
(ds / "data.yaml").write_text(f"path: {ds}\ntrain: images/train\nval: images/val\nnames:\n  0: person\n")
print(stats, "| data.yaml:\n" + (ds / "data.yaml").read_text())

In [ ]:
# Estadísticas del dataset: SIEMPRE míralas antes de entrenar
labels = [np.loadtxt(p, ndmin=2) for p in (ds / "labels" / "train").glob("*.txt") if p.stat().st_size]
L = np.concatenate(labels)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist([len(l) for l in labels], bins=20); ax[0].set_title("objetos por imagen")
ax[1].hist(L[:, 4] * 1080, bins=30); ax[1].set_title("altura de caja (px)")
ax[2].scatter(L[:, 1], L[:, 2], s=2, alpha=0.3); ax[2].invert_yaxis(); ax[2].set_title("posición de centros"); plt.show()

In [ ]:
student = YOLO(str(DATA / "yolo11n.pt"))
before = student.val(data=str(ds / "data.yaml"), device=DEVICE, classes=[0], verbose=False, plots=False,
                     project=str(DATA / "runs"), name="people_before", exist_ok=True)
student.train(data=str(ds / "data.yaml"), epochs=5, imgsz=640, batch=8, device=DEVICE, workers=0,
              project=str(DATA / "runs"), name="people_ft", exist_ok=True, verbose=False, plots=False)
after = student.val(data=str(ds / "data.yaml"), device=DEVICE, verbose=False, plots=False,
                    project=str(DATA / "runs"), name="people_after", exist_ok=True)
print(f"yolo11n vs pseudo-GT  ANTES: mAP50-95={before.box.map:.3f}  DESPUÉS fine-tune: mAP50-95={after.box.map:.3f}")

> Nota honesta: con pseudo-etiquetas el alumno está limitado por el profesor y comparte sus
> errores. Se usa para arrancar rápido; para producción hay que **revisar etiquetas a mano** y evaluar contra
> un **test anotado por humanos**.

## 7. Desequilibrio de clases y augmentations

- Deporte: cientos de cajas de jugador por cada balón. Soluciones: sobremuestrear imágenes con la clase rara,
  pérdidas ponderadas / focal, modelo específico para el balón (o a mayor resolución), copy-paste augmentation.
- Augmentations útiles en vídeo deportivo: cambios de color/brillo (HSV), escala, flip horizontal (¡cuidado si
  la clase depende del lado!), **motion blur**, mosaic, recortes (*random crop*). Evitar las que no ocurren en
  producción (rotar 90° una cámara fija).
- Para tracking, además: datos con **oclusiones y cruces**; si no, el detector nunca aprende a separar a dos
  personas solapadas.